# Domain Modules — Condensed Matter, Optimization, Finance & ML

This notebook provides a deep dive into the four domain-specific modules
in [qdk-pythonic](https://github.com/brycewestheimer/qdk-pythonic). Each
domain maps a scientific problem directly to quantum circuits — domain
scientists work in their own language (lattices, Hamiltonians, strike
prices, feature vectors) and the package handles circuit construction,
Q# generation, and resource estimation under the hood.

See [qdk_pythonic_api_demo.ipynb](qdk_pythonic_api_demo.ipynb) for an
introduction to the core circuit-builder API.

In [ ]:
%pip install qsharp>=1.25 git+https://github.com/brycewestheimer/qdk-pythonic.git

In [ ]:
import qsharp

from qdk_pythonic import Circuit, qft, inverse_qft
from qdk_pythonic.domains.condensed_matter import (
    Chain, SquareLattice, HexagonalLattice,
    IsingModel, HeisenbergModel, HubbardModel,
    simulate_dynamics,
)
from qdk_pythonic.domains.optimization import MaxCut, QUBO, TSP, QAOA
from qdk_pythonic.domains.finance import (
    LogNormalDistribution, EuropeanCallOption, QuantumAmplitudeEstimation,
)
from qdk_pythonic.domains.ml import (
    AngleEncoding, AmplitudeEncoding, QuantumKernel, VariationalClassifier,
)
from qdk_pythonic.domains.common import HardwareEfficientAnsatz, PauliHamiltonian

print("Imports successful.")

---

## Condensed Matter

The `condensed_matter` module provides lattice geometries (`Chain`,
`SquareLattice`, `HexagonalLattice`) and spin models (`IsingModel`,
`HeisenbergModel`, `HubbardModel`) that convert directly to Pauli
Hamiltonians and Trotter time-evolution circuits.

### Heisenberg XXZ Model on a Chain

The anisotropic Heisenberg model with $J_x = J_y = 1.0$, $J_z = 0.5$
on a 6-site chain. The Hamiltonian is:

$$H = \sum_{\langle i,j \rangle} \left[ J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right]$$

In [ ]:
model = HeisenbergModel(Chain(6), Jx=1.0, Jy=1.0, Jz=0.5)
ham = model.to_hamiltonian()
circuit = simulate_dynamics(model, time=1.0, steps=10)

print(f"Lattice: Chain(6), Edges: {Chain(6).edges}")
print(f"Hamiltonian terms: {len(ham)}")
print(f"Qubits: {circuit.qubit_count()}")
print(f"Gates:  {circuit.total_gate_count()}")
print(f"Depth:  {circuit.depth()}")
print(f"Gate breakdown: {circuit.gate_count()}")

### Hubbard Model — Jordan-Wigner Mapping

The Fermi-Hubbard model uses $2N$ qubits for $N$ lattice sites (one set for
spin-up, one for spin-down). The Jordan-Wigner transformation maps
fermionic operators to Pauli strings with Z-string parity phases.

In [ ]:
hubbard = HubbardModel(Chain(4), t=1.0, U=2.0)
ham_hub = hubbard.to_hamiltonian()
circuit_hub = simulate_dynamics(hubbard, time=0.5, steps=5)

print(f"Lattice sites: {Chain(4).num_sites}")
print(f"Qubits (2N):   {circuit_hub.qubit_count()}")
print(f"Hamiltonian terms: {len(ham_hub)}")
print(f"Gates:  {circuit_hub.total_gate_count()}")
print(f"Depth:  {circuit_hub.depth()}")

### 2D Lattice Geometries

The same models work on 2D lattices. Circuit resources grow with the
number of edges (nearest-neighbor interactions), not just sites.

In [ ]:
# Square lattice with Ising model
sq = SquareLattice(3, 3)
ising_2d = IsingModel(sq, J=1.0, h=0.5)
circ_sq = simulate_dynamics(ising_2d, time=0.5, steps=5)

print(f"SquareLattice(3,3): {sq.num_sites} sites, {len(sq.edges)} edges")
print(f"  Qubits: {circ_sq.qubit_count()}, Gates: {circ_sq.total_gate_count()}, Depth: {circ_sq.depth()}")

# Hexagonal lattice with Heisenberg model
hx = HexagonalLattice(2, 2)
heisen_hex = HeisenbergModel(hx, Jx=1.0, Jy=1.0, Jz=1.0)
circ_hex = simulate_dynamics(heisen_hex, time=0.5, steps=5)

print(f"\nHexagonalLattice(2,2): {hx.num_sites} sites, {len(hx.edges)} edges")
print(f"  Qubits: {circ_hex.qubit_count()}, Gates: {circ_hex.total_gate_count()}, Depth: {circ_hex.depth()}")

### Scaling Comparison

How do circuit resources scale across lattice types for the same model
(Ising, $J=1.0$, $h=0.5$, time=1.0, 10 Trotter steps)?

In [ ]:
print(f"{'Lattice':<25} {'Sites':>5} {'Edges':>5} {'Qubits':>6} {'Gates':>8} {'Depth':>6}")
print("-" * 60)

lattices = [
    ("Chain(4)", Chain(4)),
    ("Chain(8)", Chain(8)),
    ("Chain(12)", Chain(12)),
    ("SquareLattice(2,2)", SquareLattice(2, 2)),
    ("SquareLattice(3,3)", SquareLattice(3, 3)),
    ("HexagonalLattice(2,2)", HexagonalLattice(2, 2)),
]

for name, lat in lattices:
    model = IsingModel(lat, J=1.0, h=0.5)
    circ = simulate_dynamics(model, time=1.0, steps=10)
    print(
        f"{name:<25} {lat.num_sites:>5} {len(lat.edges):>5} "
        f"{circ.qubit_count():>6} {circ.total_gate_count():>8} {circ.depth():>6}"
    )

---

## Combinatorial Optimization

The `optimization` module provides `MaxCut`, `QUBO`, and `TSP` problem
classes that convert to Ising Hamiltonians, plus a `QAOA` circuit builder
that constructs variational optimization circuits.

### MaxCut on a Triangle Graph

The simplest MaxCut instance: a 3-node triangle with cost Hamiltonian

$$C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2}$$

In [ ]:
triangle = MaxCut(edges=[(0, 1), (1, 2), (2, 0)], n_nodes=3)
ham_mc = triangle.to_hamiltonian()

print(f"Nodes: {triangle.n_nodes}, Edges: {triangle.edges}")
print(f"Hamiltonian terms: {len(ham_mc)}")
for term in ham_mc.terms:
    print(f"  coeff={term.coeff:+.1f}  ops={term.pauli_ops}")

### QUBO Formulation

A Quadratic Unconstrained Binary Optimization problem with explicit
$Q$ matrix, converted to an Ising Hamiltonian via $x_i = (1 - Z_i)/2$.

In [ ]:
Q = {(0, 0): -1.0, (1, 1): -2.0, (0, 1): 0.5, (1, 2): 1.0, (2, 2): -1.5}
qubo = QUBO(Q=Q, n_vars=3)
ham_qubo = qubo.to_hamiltonian()

print(f"QUBO variables: {qubo.n_vars}")
print(f"Q matrix entries: {qubo.Q}")
print(f"Ising Hamiltonian terms: {len(ham_qubo)}")
for term in ham_qubo.terms:
    print(f"  coeff={term.coeff:+.4f}  ops={term.pauli_ops}")

### Traveling Salesman Problem (3 cities)

TSP uses one-hot encoding with $N^2$ binary variables. For 3 cities
this requires $3^2 = 9$ qubits.

In [ ]:
distances = [
    [0.0, 1.0, 2.0],
    [1.0, 0.0, 1.5],
    [2.0, 1.5, 0.0],
]
tsp = TSP(distances=distances)
ham_tsp = tsp.to_hamiltonian()
qubo_tsp = tsp.to_qubo()

print(f"Cities: {tsp.n_cities}")
print(f"QUBO variables: {qubo_tsp.n_vars}")
print(f"QUBO nonzero Q entries: {len(qubo_tsp.Q)}")
print(f"Ising Hamiltonian terms: {len(ham_tsp)}")

### QAOA Circuit and Scaling Analysis

QAOA builds layered circuits from the cost Hamiltonian and a mixer.
Resources grow with the number of layers $p$.

In [ ]:
problem = MaxCut(edges=[(0, 1), (1, 2), (2, 0)], n_nodes=3)

print(f"{'p':>3} {'Params':>6} {'Gates':>8} {'Depth':>6}")
print("-" * 28)

for p in [1, 2, 3, 4]:
    qaoa = QAOA(problem.to_hamiltonian(), p=p)
    gamma = [0.5] * p
    beta = [0.7] * p
    circ = qaoa.to_circuit(gamma=gamma, beta=beta)
    print(
        f"{p:>3} {qaoa.num_parameters:>6} "
        f"{circ.total_gate_count():>8} {circ.depth():>6}"
    )

In [ ]:
# Inspect the cost Hamiltonian structure
ham = problem.to_hamiltonian()
print(f"Qubit count: {ham.qubit_count()}")
print(f"Qubit indices: {ham.qubit_indices()}")
print()
for i, term in enumerate(ham.terms):
    print(f"  Term {i}: coeff={term.coeff:+.2f}  paulis={term.pauli_ops}")

---

## Quantum Finance

The `finance` module provides `LogNormalDistribution` for price modeling,
`EuropeanCallOption` for payoff encoding, and `QuantumAmplitudeEstimation`
for QPE-based amplitude estimation.

### Option Pricing Circuit

A European call option with strike $K = 1.0$ on a log-normal price
distribution with $\mu = 0.05$, $\sigma = 0.2$, discretized into
$2^4 = 16$ bins over $[0.5, 2.0]$.

In [ ]:
dist = LogNormalDistribution(mu=0.05, sigma=0.2, n_qubits=4, bounds=(0.5, 2.0))

print(f"Distribution bins: {len(dist.bin_values())}")
print(f"Bin range: [{dist.bin_values()[0]:.3f}, {dist.bin_values()[-1]:.3f}]")
print(f"First 5 bin midpoints: {[f'{v:.3f}' for v in dist.bin_values()[:5]]}")

In [ ]:
option = EuropeanCallOption(strike=1.0, distribution=dist)
pricing_circuit = option.to_circuit(n_estimation_qubits=4)

print(f"Qubits:     {pricing_circuit.qubit_count()}")
print(f"Gates:      {pricing_circuit.total_gate_count()}")
print(f"Depth:      {pricing_circuit.depth()}")
print(f"Gate types: {pricing_circuit.gate_count()}")

### Amplitude Estimation Directly

The `QuantumAmplitudeEstimation` class can be used independently with
any state preparation and oracle circuit — the finance module uses it
internally.

In [ ]:
state_prep = dist.to_state_prep().to_circuit()
oracle = option.payoff_oracle()

print(f"State prep qubits: {state_prep.qubit_count()}, gates: {state_prep.total_gate_count()}")
print(f"Oracle qubits:     {oracle.qubit_count()}, gates: {oracle.total_gate_count()}")

qae = QuantumAmplitudeEstimation(
    state_prep=state_prep,
    oracle=oracle,
    n_estimation_qubits=4,
)
full_circuit = qae.to_circuit()
print(f"\nFull QAE circuit:")
print(f"  Qubits: {full_circuit.qubit_count()}")
print(f"  Gates:  {full_circuit.total_gate_count()}")
print(f"  Depth:  {full_circuit.depth()}")

---

## Quantum Machine Learning

The `ml` module provides data encoding schemes (`AngleEncoding`,
`AmplitudeEncoding`), `QuantumKernel` for kernel estimation, and
`VariationalClassifier` for parameterized classification circuits.

### Feature Encoding

`AngleEncoding` maps each feature $x_i$ to $R_y(x_i)$ on qubit $i$.
`AmplitudeEncoding` loads a normalized vector into qubit amplitudes.

In [ ]:
# Angle encoding: 1 qubit per feature
angle_enc = AngleEncoding(n_features=4)
circ_angle = angle_enc.to_circuit(data=[0.1, 0.2, 0.3, 0.4])

print("AngleEncoding(n_features=4):")
print(f"  Qubits: {circ_angle.qubit_count()}")
print(f"  Gates:  {circ_angle.gate_count()}")
print(circ_angle.draw())

In [ ]:
# Amplitude encoding: 2 qubits encode 4 amplitudes
amp_enc = AmplitudeEncoding(n_qubits=2)
circ_amp = amp_enc.to_circuit(data=[0.5, 0.5, 0.5, 0.5])

print("AmplitudeEncoding(n_qubits=2):")
print(f"  Qubits: {circ_amp.qubit_count()}")
print(f"  Gates:  {circ_amp.gate_count()}")
print(circ_amp.draw())

### Quantum Kernel

The quantum kernel $k(x, y) = |\langle \phi(x) | \phi(y) \rangle|^2$ is
computed via the compute-uncompute method: apply $U(x)$, then $U^\dagger(y)$,
then measure.

In [ ]:
encoding = AngleEncoding(n_features=4)
kernel = QuantumKernel(encoding)
kernel_circ = kernel.to_circuit(
    x=[0.1, 0.2, 0.3, 0.4],
    y=[0.5, 0.6, 0.7, 0.8],
)

print(f"Kernel circuit:")
print(f"  Qubits: {kernel_circ.qubit_count()}")
print(f"  Gates:  {kernel_circ.total_gate_count()}")
print(f"  Depth:  {kernel_circ.depth()}")
print(f"  Gate breakdown: {kernel_circ.gate_count()}")

### Variational Classifier

Combines `AngleEncoding` with a `HardwareEfficientAnsatz` (alternating
$R_y$/$R_z$ rotation layers and CX entangling layers) and a single-qubit
measurement for binary classification.

In [ ]:
ansatz = HardwareEfficientAnsatz(n_qubits=4, depth=2)
classifier = VariationalClassifier(
    encoding=AngleEncoding(n_features=4),
    ansatz=ansatz,
)

# Random parameter values for demonstration
params = [0.1] * ansatz.num_parameters
classifier_circ = classifier.to_circuit(
    data=[0.1, 0.2, 0.3, 0.4],
    params=params,
)

print(f"Ansatz parameters: {ansatz.num_parameters}")
print(f"Classifier circuit:")
print(f"  Qubits: {classifier_circ.qubit_count()}")
print(f"  Gates:  {classifier_circ.total_gate_count()}")
print(f"  Depth:  {classifier_circ.depth()}")

In [ ]:
# Generated Q# for the classifier circuit
print(classifier_circ.to_qsharp())

---

## Notes

- All domain objects produce standard `Circuit` instances — `draw()`,
  `to_qsharp()`, `to_openqasm()`, `estimate()`, and `run()` work
  uniformly across domains.
- Trotter step count and order control the accuracy-vs-resources tradeoff
  for time-evolution circuits.
- The finance module's QAE circuit grows exponentially with
  `n_estimation_qubits` — use 4–6 for demonstrations, more for
  production accuracy.
- `VariationalClassifier` measures only qubit 0 — the classification
  output is the measurement probability.